Name: Muhammad Raziq Bin Sufian

Id: 24006626

## 1) Storage Method Justification

### Method Chosen: InfluxDB (Time-Series Database)

Justification: A digital twin is essentially a live model whose state changes continuously as new sensor readings arrive. InfluxDB is purpose-built for this — every write is a timestamped point, so:

- **Current state retrieval** is a `last()` query on a tag (e.g. `twin_id`), which is fast and doesn't require a separate "current value" table.
- **Historical/streaming sensor data** is stored natively as time series, with built-in range filtering and aggregation (`mean()`, `count()`, downsampling) that a plain relational table or flat file would need extra code to replicate.
- It is schema-flexible (fields can differ between measurements/twins) which suits a digital twin with heterogeneous object types (server racks vs. delivery bots).

Compared to a flat file (no querying, easy to corrupt on concurrent writes) or a plain relational DB (no native time-bucketing), InfluxDB is the better fit for both the "current state" and "sensor time stream" halves of this assignment using a single store.

## 2) Digital Twin State Variables & Sensor Data Justification

We reuse the two digital twin object types from Assignment 1:

**Server Rack (`Server_Rack_1`)**
- *State variables* (measurement `twin_state`): `operational_status` (online / maintenance / offline) and `alert_level` (normal / warning) — these describe the *condition* of the twin, used for automation/alerting logic, and don't need to be re-derived from raw numbers every time they're needed.
- *Sensor stream* (measurement `sensor_stream`): `temperature_c`, `humidity_pct`, `power_usage_kw` — raw, high-frequency telemetry used for trend analysis and threshold checks over time.

**Delivery Bot (`Delivery_Bot_1`)**
- *State variables*: `status` (idle / moving / charging) and `battery_pct` — the bot's current operating mode and health, needed for dispatch decisions.
- *Sensor stream*: `latitude`, `longitude`, `speed_kmh` — continuous telemetry for spatial tracking.

Splitting each twin's data into **state** (a few, more meaningful, moderately-changing variables) vs. **sensor stream** (frequent raw readings) mirrors how real digital twins are queried: dashboards mostly want the *latest state*, while analytics/ML pipelines want the *full history*.

## 3) Kafka Stream Manager Justification & Routing

Justification: Kafka sits between the data-generating sources (sensors/state updates) and the InfluxDB store. This decouples "producing" data from "persisting" it — the producer doesn't need to know anything about the database, and multiple consumers could be added later (e.g. one writes to InfluxDB, another triggers alerts) without touching the producer. Kafka also buffers bursts of messages so the storage writer isn't overwhelmed by simultaneous updates from many twins.

Protocol Route/Interface: A local Kafka broker at `localhost:9092`, topic `digitaltwin.telemetry`. State and sensor messages are both published as JSON to this topic (with `twin_id` as the message key); a background consumer thread reads them and writes each one into the correct InfluxDB measurement (`twin_state` or `sensor_stream`).

In [1]:
%pip install influxdb-client kafka-python

Defaulting to user installation because normal site-packages is not writeable
  Using cached influxdb_client-1.50.0-py3-none-any.whl.metadata (65 kB)
Using cached influxdb_client-1.50.0-py3-none-any.whl (746 kB)
Note: you may need to restart the kernel to use updated packages.


## 4) Connect to the InfluxDB store

Fill in `INFLUX_TOKEN` and `INFLUX_ORG` with your local InfluxDB values (Data > API Tokens in the InfluxDB UI). The bucket is created automatically if it doesn't exist yet.

In [1]:
# Step 1: Import dependencies and setup InfluxDB
from influxdb_client import InfluxDBClient, Point
from influxdb_client.client.write_api import SYNCHRONOUS
import json
import time
import threading
from kafka import KafkaProducer, KafkaConsumer

# --- InfluxDB Configuration ---
INFLUX_URL = "http://localhost:8086"
INFLUX_TOKEN = "fJkA8iV64KKnTXkuuzLGHLpVBIDXBAWGlm3eyZFteWjECM1Bc89nx_5ikhNerb5fg_hjAyZ6CuVo4OMAzSAJTw==" # Replace with your actual secure token
INFLUX_ORG = "raziq"
INFLUX_BUCKET = "digitaltwin"

client = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
write_api = client.write_api(write_options=SYNCHRONOUS)
query_api = client.query_api()

# Initialize Bucket
buckets_api = client.buckets_api()
if not buckets_api.find_bucket_by_name(INFLUX_BUCKET):
    buckets_api.create_bucket(bucket_name=INFLUX_BUCKET, org=INFLUX_ORG)
    print(f"Created bucket '{INFLUX_BUCKET}'")
else:
    print(f"Bucket '{INFLUX_BUCKET}' already exists")

print("InfluxDB reachable:", client.ping())

Bucket 'digitaltwin' already exists
InfluxDB reachable: True


## 7) Kafka as stream manager — producer & consumer

In [2]:
# Step 2: Define the Kafka Consumer and the parsing logic to write to InfluxDB

KAFKA_BROKER = "localhost:9092"
KAFKA_TOPIC = "digitaltwin.telemetry"
consumer_ready = threading.Event()

def write_message_to_influx(msg):
    """
    Parses the incoming JSON message and formats it into an InfluxDB Point.
    Handles both numeric sensor data and string-based state variables.
    """
    # Create a new data point using the 'measurement_name'
    point = Point(msg["measurement_name"]) \
        .tag("twin_id", msg["twin_id"]) \
        .tag("data_type", msg["type"]) # Categorize as 'sensor' or 'state'
    
    # Store numeric values (sensors) differently from string values (states)
    if isinstance(msg["value"], (int, float)):
        point.field("value_num", float(msg["value"]))
    else:
        point.field("value_str", str(msg["value"]))
        
    write_api.write(bucket=INFLUX_BUCKET, org=INFLUX_ORG, record=point)

def consume_loop():
    consumer = KafkaConsumer(
        KAFKA_TOPIC,
        bootstrap_servers=KAFKA_BROKER,
        auto_offset_reset="earliest", # <-- CHANGED BACK TO EARLIEST
        value_deserializer=lambda v: json.loads(v.decode("utf-8")),
        group_id=f"digitaltwin-storage-writer-{int(time.time())}"
    )
    # Poll once to establish connection before setting event
    consumer.poll(timeout_ms=1000)
    consumer_ready.set()
    
    while True:
        try:
            for message in consumer:
                msg = message.value
                try:
                    write_message_to_influx(msg)
                    print(f"[KAFKA CONSUMER] Stored {msg['type']} '{msg['measurement_name']}' for {msg['twin_id']} in InfluxDB")
                except Exception as e:
                    print(f"[KAFKA CONSUMER ERROR] Failed to write {msg}: {e}")
        except Exception as e:
            print(f"[KAFKA CONSUMER FATAL] Consumer loop crashed, restarting: {e}")
            time.sleep(1)

# Start consumer in a background thread
consumer_thread = threading.Thread(target=consume_loop, daemon=True)
consumer_thread.start()

if consumer_ready.wait(timeout=10):
    print("Kafka consumer subscribed and ready.")
else:
    print("Warning: consumer did not confirm readiness in time.")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_3056\1508943838.py:26: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


Kafka consumer subscribed and ready.


In [3]:
# Step 3: Produce simulated twin data to Kafka

producer = KafkaProducer(
    bootstrap_servers=KAFKA_BROKER,
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

# Define a batch of updates for the Digital Twin
digital_twin_updates = [
    # Sensor Streams
    {"twin_id": "robot_001", "type": "sensor", "measurement_name": "motor_temperature", "value": 68.5},
    {"twin_id": "robot_001", "type": "sensor", "measurement_name": "battery_voltage", "value": 24.1},
    
    # State Variables
    {"twin_id": "robot_001", "type": "state", "measurement_name": "operational_status", "value": "moving"},
    {"twin_id": "robot_001", "type": "state", "measurement_name": "current_zone", "value": "Zone_B"}
]

print("Publishing data to Kafka...")
for update in digital_twin_updates:
    producer.send(KAFKA_TOPIC, update)
producer.flush()
print("Data published successfully. Waiting for consumer to process...")

# Give the background consumer thread a moment to process the messages
time.sleep(3)

Publishing data to Kafka...
Data published successfully. Waiting for consumer to process...


In [4]:
# Step 4: Retrieve and validate data from InfluxDB

print(f"--- Retrieving Digital Twin Data from InfluxDB ---")

# Flux query to get data from the last 1 hour for robot_001
flux_query = f'''
from(bucket: "{INFLUX_BUCKET}")
  |> range(start: -1h)
  |> filter(fn: (r) => r["twin_id"] == "robot_001")
  |> keep(columns: ["_time", "_measurement", "data_type", "_field", "_value"])
  |> sort(columns: ["_time"], desc: true)
'''

result = query_api.query(org=INFLUX_ORG, query=flux_query)

# Parse and display the results
if not result:
    print("No data found. Ensure Kafka and InfluxDB are running and properly connected.")
else:
    for table in result:
        for record in table.records:
            time_stamp = record.get_time().strftime('%Y-%m-%d %H:%M:%S')
            data_type = record.values.get("data_type")
            measurement = record.values.get("_measurement") # <-- CHANGED TO _measurement
            value = record.get_value()
            
            print(f"[{time_stamp}] Type: {str(data_type).upper().ljust(7)} | Variable: {str(measurement).ljust(20)} | Value: {value}")

--- Retrieving Digital Twin Data from InfluxDB ---
No data found. Ensure Kafka and InfluxDB are running and properly connected.
